# 10 · Regression *and* classification in one network

"The test is on training a network for regression or classification — or even both." *Both at once* means a **multi-task** network: one shared body, two outputs, two losses added together. Nothing in it is new — it is notebooks 06 and 07 running in the same loop — but there are a few places where the two halves can trip over each other.

Data: `houses.csv` again, now with **two labels per house**: `price` (regression, k€) and `sold_fast` (binary: sold within 30 days).

---
## Theory in 60 seconds

```
                         ┌─> head_price: Linear(16, 1)            -> MSE          (target: float (batch, 1))
x ─> shared trunk (MLP) ─┤
                         └─> head_sold:  Linear(16, 1) -> logit   -> BCEWithLogits (target: float (batch, 1))

total loss = loss_price + λ · loss_sold          one backward(), one optimizer.step()
```

- The **trunk** learns features useful for both tasks; each **head** is just the output layer you already know for its task.
- `backward()` on the *sum* sends gradients from both losses into the shared trunk — no special machinery.
- **Loss scale matters.** The optimiser minimises the sum; a loss that is numerically 10 000× larger dominates completely. Bring the losses to comparable size: standardise the regression target and/or weight the terms with `λ`.
- Each task keeps its own metric (RMSE in k€; accuracy), and predictions for the regression head must be **un-standardised** before you report them.

In [ ]:
import sys; sys.path.insert(0, "..")
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from dnd_utils import check, load_houses

d = load_houses()
X_train, y_train, s_train = d["X_train"], d["y_train"], d["s_train"]     # (720, 13), price (720,), sold_fast (720,) 0.0/1.0
X_val, y_val, s_val = d["X_val"], d["y_val"], d["s_val"]
X_test, y_test, s_test = d["X_test"], d["y_test"], d["s_test"]
print(X_train.shape, y_train[:4], s_train[:8], "| fraction sold fast:", s_train.mean().round(3))

---
## Exercise 10.1 · How different are the two loss scales?

Before any training: an untrained network outputs roughly 0 for the price and a logit of roughly 0 for `sold_fast`. Compute what the two losses are in that situation:

- `mse_at_zero`: MSE of predicting 0 for every training price,
- `bce_at_zero`: BCE of predicting logit 0 (p = 0.5) for every training label,
- `ratio = mse_at_zero / bce_at_zero`.

In [ ]:
mse_at_zero = None
bce_at_zero = None
ratio = None

In [ ]:
check("mse_at_zero ≈ 50 000", lambda: abs(mse_at_zero - float((y_train.astype(np.float64) ** 2).mean())) < 1)
check("bce_at_zero = ln 2", lambda: abs(bce_at_zero - np.log(2)) < 1e-6)
check("ratio", lambda: abs(ratio - mse_at_zero / np.log(2)) < 1)

If you simply trained on `MSE + BCE` with these raw targets, what would the network do about the classification task?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 10.2 · Standardise the regression target

Compute `y_mean` and `y_std` from the **training** prices and create `yz_train`, `yz_val`, `yz_test` (standardised prices). Then repeat the computation of 10.1 for the standardised target: `mse_z_at_zero`.

In [ ]:
y_mean = None
y_std = None
yz_train = yz_val = yz_test = None
mse_z_at_zero = None

In [ ]:
check("y_mean, y_std from the training set", lambda: abs(y_mean - d["y_train"].mean()) < 1e-3 and abs(y_std - d["y_train"].std()) < 1e-3)
check("val/test use the training statistics", lambda: np.allclose(yz_val, (d["y_val"] - d["y_train"].mean()) / d["y_train"].std(), atol=1e-5) and np.allclose(yz_test, (d["y_test"] - d["y_train"].mean()) / d["y_train"].std(), atol=1e-5))
check("MSE of predicting 0 is now exactly 1 (why?)", lambda: abs(mse_z_at_zero - 1.0) < 1e-3)

With a standardised target, "predict 0" **is** the mean-baseline and its MSE is the variance, i.e. 1. A convenient side effect: the standardised MSE reads directly as "fraction of the variance not explained" (`1 − R²`). Both losses now start near 1 and 0.69 — comparable.

---
## Exercise 10.3 · A dataset with two labels

Write `TwoTaskDataset(x, y_price, y_sold)`: `__getitem__` returns **three** tensors so that a loader yields `(batch, 13)`, `(batch, 1)`, `(batch, 1)`, all `float32`. Create the three loaders (batch size 32) using the **standardised** prices.

In [ ]:
class TwoTaskDataset(Dataset):
    pass

loader_train = loader_val = loader_test = None

In [ ]:
check("a batch is a triple", lambda: len(next(iter(loader_train))) == 3)
check("shapes (32, 13), (32, 1), (32, 1)", lambda: [tuple(t.shape) for t in next(iter(loader_train))] == [(32, 13), (32, 1), (32, 1)])
check("all float32", lambda: all(t.dtype == torch.float32 for t in next(iter(loader_train))))
check("the price target is standardised", lambda: abs(torch.cat([b[1] for b in loader_train]).mean().item()) < 1e-3)
check("the sold target is 0/1", lambda: set(torch.cat([b[2] for b in loader_train]).unique().tolist()) == {0.0, 1.0})

---
## Exercise 10.4 · The two-headed model

Write `TwoHeadMLP(n_inputs)`:

- `trunk`: `n_inputs → 32 → 16`, ReLU after both layers,
- `head_price`: `16 → 1` (no activation),
- `head_sold`: `16 → 1` (a logit, no activation),
- `forward(x)` returns the **tuple** `(price_pred, sold_logit)`.

How many parameters does it have (by hand first)?

In [ ]:
n_params_by_hand = None

class TwoHeadMLP(nn.Module):
    pass

In [ ]:
_out = None
try: _out = TwoHeadMLP(13)(torch.randn(5, 13))
except Exception: pass
check("forward returns two tensors", lambda: isinstance(_out, tuple) and len(_out) == 2)
check("each of shape (5, 1)", lambda: _out[0].shape == (5, 1) and _out[1].shape == (5, 1))
check("1010 parameters", lambda: sum(p.numel() for p in TwoHeadMLP(13).parameters()) == 1010 == n_params_by_hand)
check("both heads are registered sub-modules", lambda: {"head_price", "head_sold", "trunk"} <= {n for n, _ in TwoHeadMLP(13).named_children()})

---
## Exercise 10.5 · Train on the combined loss

Train a fresh `TwoHeadMLP` for 80 epochs with `Adam(lr=0.003)` on

`loss = MSE(price_pred, price_z) + LAMBDA · BCEWithLogits(sold_logit, sold)`, with `LAMBDA = 1.0`.

Per epoch record, on the **validation** set: `val_total`, `val_mse`, `val_bce`, `val_acc` (lists), and `train_total` on the training set.

<details><summary>Hint</summary>

A loop over the loader now unpacks three things: `for xb, yp, ys in loader:`. The model call unpacks two: `price_pred, sold_logit = model(xb)`. Everything else is the loop you know. Only **one** `backward()` — on the sum.
</details>

In [ ]:
LAMBDA = 1.0
torch.manual_seed(0)
model = None
train_total, val_total, val_mse, val_bce, val_acc = [], [], [], [], []

In [ ]:
check("80 epochs recorded in all five lists", lambda: all(len(l) == 80 for l in (train_total, val_total, val_mse, val_bce, val_acc)))
check("total = mse + LAMBDA * bce", lambda: len(val_total) == 80 and np.allclose(np.array(val_total), np.array(val_mse) + LAMBDA * np.array(val_bce), atol=1e-4))
check("the regression task was learned (standardised MSE < 0.08)", lambda: val_mse[-1] < 0.08)
check("the classification task was learned (best validation BCE clearly below ln 2)", lambda: min(val_bce) < 0.6)

Plot three panels: total loss (train and validation), the two validation loss components, and the validation accuracy.

In [ ]:
# your plot

Look at the middle panel. The two tasks do **not** behave the same over time. Describe what each validation component does after roughly epoch 15–20, why that can happen, and name two things you could do about it.

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 10.6 · Evaluate both tasks on the test set — in real units

Compute on the full test set:

- `price_pred_keur`: predicted prices **in k€**, shape `(240, 1)` (undo the standardisation!)
- `test_rmse_keur`: RMSE in k€
- `baseline_rmse_keur`: RMSE of always predicting the training mean price
- `test_acc` for `sold_fast`, and `baseline_acc` (majority class of the training set)

In [ ]:
price_pred_keur = None
test_rmse_keur = None
baseline_rmse_keur = None
test_acc = None
baseline_acc = None

In [ ]:
check("price_pred_keur is in k€ (mean around 200)", lambda: price_pred_keur.shape == (240, 1) and 150 < price_pred_keur.mean().item() < 260)
check("RMSE consistent with the predictions", lambda: abs(test_rmse_keur - float(np.sqrt(((price_pred_keur[:, 0].numpy() - y_test) ** 2).mean()))) < 1e-2)
check("RMSE far below the baseline", lambda: test_rmse_keur < 0.3 * baseline_rmse_keur)
check("baseline_acc", lambda: abs(baseline_acc - (s_test == 0).mean()) < 1e-9)
check("accuracy above the baseline", lambda: test_acc > baseline_acc + 0.03)

A shortcut worth knowing: the RMSE in k€ equals the RMSE in standardised units times `y_std` (the mean shift cancels in the difference). What is the corresponding rule for the **MSE**?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 10.7 · The other way to build it: one output layer, sliced

Instead of two head modules you can use a single final `Linear(16, 2)` and treat column 0 as the price and column 1 as the logit. Write `OneLayerTwoOutputs(n_inputs)` whose `forward` returns a single tensor of shape `(batch, 2)`, and `combined_loss(out, yp, ys, lam)` that slices it and returns the same combined loss as before.

Careful with the slicing: which of `out[:, 0]` and `out[:, :1]` has the shape that matches a `(batch, 1)` target?

In [ ]:
class OneLayerTwoOutputs(nn.Module):
    pass

def combined_loss(out, yp, ys, lam):
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
_o = torch.randn(32, 2); _yp = torch.randn(32, 1); _ys = (torch.rand(32, 1) < 0.3).float()
check("model output shape (5, 2)", lambda: OneLayerTwoOutputs(13)(torch.randn(5, 13)).shape == (5, 2))
check("combined_loss value", lambda: torch.allclose(combined_loss(_o, _yp, _ys, 0.5), ((_o[:, 0] - _yp[:, 0]) ** 2).mean() + 0.5 * nn.BCEWithLogitsLoss()(_o[:, 1], _ys[:, 0]), atol=1e-6))
check("scalar with a graph", lambda: (lambda l: l.ndim == 0 and l.requires_grad)(combined_loss(OneLayerTwoOutputs(13)(torch.randn(32, 13)), _yp, _ys, 1.0)))

The two designs are mathematically the same network (a `Linear(16, 2)` *is* two `Linear(16, 1)` stacked). The same slicing idea covers "regression + **multi-class**": a final `Linear(16, 1 + k)`, with `out[:, :1]` for the regression value and `out[:, 1:]` as the `k` class logits.

---
## Exercise 10.8 · Regression + multi-class: getting labels, shapes and dtypes right

Suppose the second task is not `sold_fast` but a **price tier** with three classes, defined from the price like the categories in the homework:

- class 0: price below 150 k€, class 1: 150 ≤ price < 250, class 2: price ≥ 250.

**a)** Create `tier_train` from `y_train` (vectorised, dtype `int64`).

**b)** For the given batch of outputs `out` (shape `(batch, 4)`: 1 regression value + 3 class logits) compute `loss_reg`, `loss_cls` and `tier_pred` with the correct slices, target shapes and dtypes.

(*Think:* would it make sense to actually train this pair of tasks together? See the question below.)

In [ ]:
tier_train = None

torch.manual_seed(0)
out = torch.randn(32, 4)                                         # pretend model output for the first 32 training houses
yz_b = torch.tensor(yz_train[:32], dtype=torch.float32)          # (32,)  standardised price

loss_reg = None
loss_cls = None
tier_pred = None      # (32,) predicted class indices

In [ ]:
check("tier_train", lambda: tier_train.dtype == np.int64 and (tier_train == np.digitize(y_train, [150, 250])).all())
check("loss_reg", lambda: torch.allclose(loss_reg, ((out[:, 0] - yz_b) ** 2).mean(), atol=1e-6))
check("loss_cls", lambda: torch.allclose(loss_cls, nn.CrossEntropyLoss()(out[:, 1:4], torch.tensor(np.digitize(y_train[:32], [150, 250]))), atol=1e-6))
check("tier_pred", lambda: tier_pred.shape == (32,) and torch.equal(tier_pred, out[:, 1:].argmax(dim=1)))

If the tier is computed **from the price**, which columns must *not* be among the network's inputs — and is there anything suspicious about using `price` itself as an input for predicting `sold_fast`?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Blank-page drill

New empty notebook, 35 minutes. From `d = load_houses()` (inputs `X_*`, prices `y_*`, sold-fast labels `s_*`):

1. standardise the price target with training statistics;
2. a dataset that yields three tensors, and loaders;
3. a two-headed model `13 → 32 → 16 →` {price, sold-fast logit};
4. training with the summed loss and **early stopping on the total validation loss** (patience 15, restore the best weights);
5. a plot of the training and validation total loss;
6. test RMSE **in k€** and test accuracy, each next to its baseline.

The solution notebook has this version at the end.

In [ ]:
# (do this one in a new notebook)